In [1]:
pip install wordcloud seaborn scipy


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd

In [3]:
def get_written_dates_and_any_text_file_locations_to_dataframe(root_folder):
    print(f"--- Starting Scan for metadata.json and any associated non-JSON text files ---")
    print(f"Scanning root folder: {os.path.abspath(root_folder)}\n")

    if not os.path.exists(root_folder):
        print(f"Error: The root folder '{root_folder}' does not exist at '{os.path.abspath(root_folder)}'.")
        return pd.DataFrame()

    data_for_df = []
    
    for dirpath, dirnames, filenames in os.walk(root_folder):
        metadata_filepath_in_dir = None
        text_file_path_in_dir = None
        
        non_json_files = []

        # First, identify the metadata.json and any non-JSON files in the current directory
        for filename in filenames:
            if filename.lower() == "metadata.json":
                metadata_filepath_in_dir = os.path.join(dirpath, filename)
            elif filename.lower().endswith(".txt"): # Consider anything that's not JSON as a potential text file
                non_json_files.append(os.path.join(dirpath, filename))

        # Check conditions for a valid pair
        if metadata_filepath_in_dir:
            if len(non_json_files) == 1:
                # Exactly one non-JSON file found, assume it's the text file
                text_file_path_in_dir = non_json_files[0]
                
                print(f"Processing directory: {os.path.abspath(dirpath)}")
                print(f"  Found metadata.json: {metadata_filepath_in_dir}")
                print(f"  Found associated text file: {text_file_path_in_dir}")

                extracted_written_date = None
                try:
                    with open(metadata_filepath_in_dir, 'r', encoding='utf-8') as f:
                        metadata = json.load(f)
                        if 'written_date' in metadata:
                            extracted_written_date = metadata['written_date']
                            print(f"  Extracted 'written_date': '{extracted_written_date}'")
                        else:
                            print(f"  Warning: 'written_date' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")
                except json.JSONDecodeError as e:
                    print(f"  Error: Could not parse JSON from {metadata_filepath_in_dir}: {e}. Setting 'written_date' to None.")
                except Exception as e:
                    print(f"  An unexpected error occurred reading {metadata_filepath_in_dir}: {e}. Setting 'written_date' to None.")
                
                # Add this pair to our DataFrame data
                data_for_df.append({
                    'written_date': extracted_written_date,
                    'text_file_location': os.path.abspath(text_file_path_in_dir)
                })
                
            elif len(non_json_files) == 0:
                print(f"  Note: Found metadata.json in {os.path.abspath(dirpath)} but NO non-JSON text file found. Skipping this directory.")
            else: # len(non_json_files) > 1
                print(f"  Warning: Found metadata.json in {os.path.abspath(dirpath)} but MULTIPLE non-JSON files ({non_json_files}). Cannot determine which is the text file. Skipping this directory.")
        # No else for when metadata.json is not found at all, to keep output cleaner

    print(f"\n--- Scan Complete ---")
    print(f"Total valid pairs (metadata.json + single non-JSON text file): {len(data_for_df)}")
    
    df = pd.DataFrame(data_for_df)

    if df.empty:
        print("No valid pairs of metadata.json and single non-JSON text file found to create a DataFrame.")
    else:
        print("\n--- Resulting DataFrame Head ---")
        print(df.head())
        print(f"\nDataFrame has {len(df)} rows and {len(df.columns)} columns.")
    
    return df

In [4]:
root_directory_to_scan = "data/OCR_Final"

list_of_file_locations = get_written_dates_and_any_text_file_locations_to_dataframe(root_directory_to_scan)

--- Starting Scan for metadata.json and any associated non-JSON text files ---
Scanning root folder: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final

Processing directory: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව
  Found metadata.json: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/metadata.json
  Found associated text file: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව.txt
  Extracted 'written_date': '1187 - 1225'
Processing directory: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final/නිදහසේ මන්ත්‍රය
  Found metadata.json: data/OCR_Final/නිදහසේ මන්ත්‍රය/metadata.json
  Found associated text file: data/OCR_Final/නිදහසේ මන්ත්‍රය/නිදහසේ මන්ත්‍රය.txt
  Extracted 'written_date': '1901 - 1938'
Processing directory: /Users/nevidujayatilleke/Documents/MSC - Res

In [5]:
list_of_file_locations

,written_date,text_file_location
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...
1,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...
2,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...
3,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...
4,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...
5,1850 - 1903,/Users/nevidujayatilleke/Documents/MSC - Resea...
6,1220 - 1293,/Users/nevidujayatilleke/Documents/MSC - Resea...
7,1854 - 1899,/Users/nevidujayatilleke/Documents/MSC - Resea...
8,1706 - 1739,/Users/nevidujayatilleke/Documents/MSC - Resea...
9,1857 - 1922,/Users/nevidujayatilleke/Documents/MSC - Resea...


In [6]:
list_of_file_locations['text_file_location'][1]

'/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/Surya_OCR/data/OCR_Final/නිදහසේ මන්ත්\u200dරය/නිදහසේ මන්ත්\u200dරය.txt'

In [7]:
import numpy as np

In [8]:
list_of_file_locations['start_date'] = np.nan
list_of_file_locations['end_date'] = np.nan

In [9]:
for index, row in list_of_file_locations.iterrows():
    date_str = str(row['written_date']) # Ensure it's treated as a string
    
    start_year = np.nan
    end_year = np.nan

    if ' - ' in date_str:
        # It's a range
        try:
            start_year_str, end_year_str = date_str.split(' - ')
            start_year = int(start_year_str.strip())
            end_year = int(end_year_str.strip())
        except ValueError:
            print(f"Warning: Could not parse range '{date_str}' at index {index}. Setting to NaN.")
    else:
        # It's a single year or potentially unparseable
        try:
            single_year = int(date_str.strip())
            start_year = single_year
            end_year = single_year # Duplicate for single date
        except ValueError:
            print(f"Warning: Could not parse single year/date '{date_str}' at index {index}. Setting to NaN.")
            
    # Assign the parsed (or NaN) values back to the DataFrame
    list_of_file_locations.at[index, 'start_date'] = start_year
    list_of_file_locations.at[index, 'end_date'] = end_year


In [10]:
list_of_file_locations.head()

,written_date,text_file_location,start_date,end_date
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,1187.0,1225.0
1,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901.0,1938.0
2,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1944.0,1944.0
3,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1303.0,1333.0
4,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...,1825.0,1905.0


In [11]:
def get_century(year):
    """Calculates the century for a given year."""
    if pd.isna(year):
        return np.nan
    return (int(year) - 1) // 100 + 1

In [12]:
list_of_file_locations['end_century'] = list_of_file_locations['end_date'].apply(get_century)

In [13]:
list_of_file_locations.head()

,written_date,text_file_location,start_date,end_date,end_century
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,1187.0,1225.0,13
1,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901.0,1938.0,20
2,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1944.0,1944.0,20
3,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1303.0,1333.0,14
4,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...,1825.0,1905.0,20


In [14]:
import re

In [15]:
def count_clean_words(path):
    try:
        with open(path, 'r', encoding='utf-8') as file:
            text = file.read()

        # Keep Sinhala + Latin characters + whitespace, remove others
        cleaned = re.sub(r'[^\u0D80-\u0DFFa-zA-Z\s]', '', text)

        # Tokenize by whitespace
        words = cleaned.split()
        # print(words)
        return len(words)

    except Exception as e:
        print(f"Error reading {path}: {e}")
        return 0

In [16]:
list_of_file_locations['word_count_no_punct_num'] = list_of_file_locations['text_file_location'].apply(count_clean_words)

In [17]:
list_of_file_locations['word_count_no_punct_num']

0     1126
1     1280
2     1661
3     1677
4     1283
5      973
6     2022
7     1886
8     1142
9      979
10    1228
11    1097
12    1547
13    1270
14    1137
15    1583
16    2242
17    1515
18    1275
19     848
20    1472
21     920
22    1410
23     843
24     925
25    1656
26    1415
27    1302
28     679
29    1148
30    1062
31    1227
32    1020
33     901
34    1474
35     988
36    1779
37    1311
38    1162
39    1626
40     811
41    1871
42    1280
43     694
44    1623
45     473
Name: word_count_no_punct_num, dtype: int64

In [18]:
list_of_file_locations['word_count_no_punct_num'].sum()

np.int64(58843)

In [19]:
def count_english_words(path):
    try:
        with open(path, 'r', encoding='utf-8') as file:
            text = file.read()

        # Keep only English letters and spaces
        cleaned = re.sub(r'[^a-zA-Z\s]', '', text)

        # Tokenize and filter out empty tokens
        words = [word for word in cleaned.split() if word.isalpha()]

        return len(words)

    except Exception as e:
        print(f"Error reading {path}: {e}")
        return 0

In [20]:
list_of_file_locations['english_word_count'] = list_of_file_locations['text_file_location'].apply(count_english_words)

In [21]:
list_of_file_locations['english_word_count']

0       0
1       0
2       0
3       0
4       1
5       0
6       0
7       0
8       0
9     630
10      0
11    147
12     13
13      0
14      0
15      0
16      0
17      1
18      3
19      0
20      0
21      0
22      0
23     24
24      0
25     12
26      0
27      0
28      0
29      0
30      0
31      0
32      0
33      0
34      0
35      0
36      0
37      0
38      0
39      0
40      0
41      1
42      0
43      0
44      0
45      0
Name: english_word_count, dtype: int64

In [22]:
list_of_file_locations['english_word_count'].sum()

np.int64(832)

In [23]:
list_of_file_locations.head(10)

,written_date,text_file_location,start_date,end_date,end_century,word_count_no_punct_num,english_word_count
0,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,1187.0,1225.0,13,1126,0
1,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901.0,1938.0,20,1280,0
2,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1944.0,1944.0,20,1661,0
3,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1303.0,1333.0,14,1677,0
4,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...,1825.0,1905.0,20,1283,1
5,1850 - 1903,/Users/nevidujayatilleke/Documents/MSC - Resea...,1850.0,1903.0,20,973,0
6,1220 - 1293,/Users/nevidujayatilleke/Documents/MSC - Resea...,1220.0,1293.0,13,2022,0
7,1854 - 1899,/Users/nevidujayatilleke/Documents/MSC - Resea...,1854.0,1899.0,19,1886,0
8,1706 - 1739,/Users/nevidujayatilleke/Documents/MSC - Resea...,1706.0,1739.0,18,1142,0
9,1857 - 1922,/Users/nevidujayatilleke/Documents/MSC - Resea...,1857.0,1922.0,20,979,630


In [24]:
all_text = ""
for path in list_of_file_locations['text_file_location']:
    try:
        with open(path, 'r', encoding='utf-8') as file:
            all_text += file.read()
    except Exception as e:
        print(f"Error reading {path}: {e}")

In [25]:
cleaned_text = re.sub(r'[^\u0D80-\u0DFF\s]', '', all_text)

In [26]:
cleaned_text

'\t\t\t\t\t\t\tධර්ම ප්රදීපිකාව\n\n\t\t\t\t\t\tනමෝ තස්ස භගවතො අරහතො\n\t\t\t\t\t\t\tසම්මා සම්බුද්ධස්ස\n\n\tඅප බුදුන් සාරාසකි  කප්සුවහස් මතුයෙහි කුලුණුනු වණින්\nයුත් මහත වු සක් වැ දිවකුරු බුදුන් හමුවැ අතට පත් නිවන් සසර සේ\nපියා සතුන් සඳහා විඳුනා සසර දුක් නිවන් සේ ගෙන භව දුර්ගයට\nවැද පැරුම් පුරා තු පුර පැමිණ දසදහස් ලෝදාහි  දෙව්බඹුන්ගේ\nඅයජ මෙන් කිඹුල්වත්හි ඉපද වැඩිවිය පැමිණ මහභිනික්මන් කොට\nපැවිජි වැ මහා වීර්යය කොට බොධිපර්යංකාරූඪ වැ මාරවිජය කොට\nසර්වඥපදප්රාප්ත වැ හුනස්නෙන් නැඟී පූර්වොත්තර දිශා භාගයෙහි\nවැඩ සිට සප්තාහයක් අනිමිසලෝචන යුගලයෙන් පුදන ලද ජය\nමහා බොධිය තත ප්රභව වූ ඵල රුහ මහා බොධිය තදවයව වූ ශාඛා\nහා බොධිය යන තුන් මහා බොධීන්ගේ වංශයෙහි සහස්රරශ්මින්\nවිභක්ත පදයන් අතුරෙහි පරිකථානුකූල පද ගෙන වර්ණනා\nකරනු ලැබේ\n\t යස්ස යනු චතුර්විධ පදයන් කෙරෙහි නාමපදයි එහි චතුර්විධ\nපද නම් කවරයත් නාම පදය ආඛ්යාත පදය උපසර්ග පදය\nනිපාතපදයි\n\nඑහි  නාම පද නම් සර්වාර්ථයන් නමන්නෙන් නාම පද නම්\nවේ එයින් කීහ\n\n\tයත්ත්රිලිංගං ත්රිවචනං සර්වාසු ච විභක්තිෂු\n\tනමත්යර්ථෙන සර්වාර්ථා නාම කවයො විදුඃ  යී\n\nක්රියාවාචක ප

In [27]:
tokens = cleaned_text.split()

In [28]:
tokens

['ධර්ම',
 'ප්රදීපිකාව',
 'නමෝ',
 'තස්ස',
 'භගවතො',
 'අරහතො',
 'සම්මා',
 'සම්බුද්ධස්ස',
 'අප',
 'බුදුන්',
 'සාරාසකි',
 'කප්සුවහස්',
 'මතුයෙහි',
 'කුලුණුනු',
 'වණින්',
 'යුත්',
 'මහත',
 'වු',
 'සක්',
 'වැ',
 'දිවකුරු',
 'බුදුන්',
 'හමුවැ',
 'අතට',
 'පත්',
 'නිවන්',
 'සසර',
 'සේ',
 'පියා',
 'සතුන්',
 'සඳහා',
 'විඳුනා',
 'සසර',
 'දුක්',
 'නිවන්',
 'සේ',
 'ගෙන',
 'භව',
 'දුර්ගයට',
 'වැද',
 'පැරුම්',
 'පුරා',
 'තු',
 'පුර',
 'පැමිණ',
 'දසදහස්',
 'ලෝදාහි',
 'දෙව්බඹුන්ගේ',
 'අයජ',
 'මෙන්',
 'කිඹුල්වත්හි',
 'ඉපද',
 'වැඩිවිය',
 'පැමිණ',
 'මහභිනික්මන්',
 'කොට',
 'පැවිජි',
 'වැ',
 'මහා',
 'වීර්යය',
 'කොට',
 'බොධිපර්යංකාරූඪ',
 'වැ',
 'මාරවිජය',
 'කොට',
 'සර්වඥපදප්රාප්ත',
 'වැ',
 'හුනස්නෙන්',
 'නැඟී',
 'පූර්වොත්තර',
 'දිශා',
 'භාගයෙහි',
 'වැඩ',
 'සිට',
 'සප්තාහයක්',
 'අනිමිසලෝචන',
 'යුගලයෙන්',
 'පුදන',
 'ලද',
 'ජය',
 'මහා',
 'බොධිය',
 'තත',
 'ප්රභව',
 'වූ',
 'ඵල',
 'රුහ',
 'මහා',
 'බොධිය',
 'තදවයව',
 'වූ',
 'ශාඛා',
 'හා',
 'බොධිය',
 'යන',
 'තුන්',
 'මහා',
 'බොධීන්ගේ',
 'වංශයෙහි',
 'සහස්රරශ්මින්',
 '

In [29]:
len(tokens)

58027

In [30]:
unique_tokens = sorted(set(tokens))

In [31]:
unique_tokens

['ං',
 'ංඑත්ථාහ',
 'ංකාරූඪව',
 'ංග',
 'ංගයකින්',
 'ංග්රහව',
 'ංඤ්ච',
 'ංඩය',
 'ඃ',
 'අ',
 'අං',
 'අංඉති',
 'අංග',
 'අංගණ',
 'අංගණාවට',
 'අංගනාවන්ගෙ',
 'අංගාන්',
 'අංගානි',
 'අංගුත්තර',
 'අංජනනිලේ',
 'අංයන',
 'අංයයි',
 'අංසක',
 'අආඉඊඋඌ',
 'අආදයො',
 'අඉඋ',
 'අඑ',
 'අක',
 'අකංසු',
 'අකණිටායෙහි',
 'අකඬලා',
 'අකප්පියං',
 'අකප්පියසයනානි',
 'අකප්පියානි',
 'අකප්පියානිසයනානි',
 'අකම්පිත්ථ',
 'අකමිකයි',
 'අකර්මක',
 'අකරොන්තෙන',
 'අකල්යල්ය',
 'අකලිෂ්ට',
 'අක්',
 'අක්කොච්ඡිමං',
 'අක්ඛන්ති',
 'අක්ඛරවන්තො',
 'අක්ඛරා',
 'අක්ඛරාපාදයො',
 'අක්ඛරාපි',
 'අක්ඛරෙසු',
 'අක්ඛෙපෙච',
 'අක්පතුල්',
 'අක්ෂ',
 'අක්ෂර',
 'අක්ෂරයන්',
 'අක්ෂරයන්ගේ',
 'අක්ෂරයෙන්',
 'අක්ෂරයෝ',
 'අක්ෂි',
 'අකා',
 'අකාර',
 'අකාරතො',
 'අකාරය',
 'අකාරයක්හු',
 'අකාරයහා',
 'අකාරයා',
 'අකාරාදයො',
 'අකාරාන්ත',
 'අකාරො',
 'අකැ',
 'අකැප',
 'අකැපපී',
 'අකීකරුවී',
 'අකුරකට',
 'අකුරු',
 'අකුරුත්',
 'අකුරුතුණ',
 'අකුරුයැයි',
 'අකුරෙකි',
 'අකුරෙන්',
 'අකුසලට',
 'අකුසලයක්',
 'අකුසල්',
 'අකුසල්හි',
 'අකුසලින්',
 'අඛණ්ඩ',
 'අඛිනන්න',
 'අඛිනන්නං',
 'අඛින්

In [32]:
len(unique_tokens)

22837

In [33]:
century_tokens = list_of_file_locations.groupby('end_century')['word_count_no_punct_num'].sum().reset_index()


In [34]:
century_tokens

,end_century,word_count_no_punct_num
0,5,1062
1,13,10932
2,14,5941
3,15,6743
4,18,3909
5,19,11251
6,20,19005


In [35]:
def extract_sinhala_tokens(path):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
        # Keep Sinhala characters and spaces
        text = re.sub(r'[^\u0D80-\u0DFF\s]', '', text)
        return text.split()
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return []

In [36]:
list_of_file_locations['tokens'] = list_of_file_locations['text_file_location'].apply(extract_sinhala_tokens)

In [37]:
list_of_file_locations['tokens'][0]

['ධර්ම',
 'ප්රදීපිකාව',
 'නමෝ',
 'තස්ස',
 'භගවතො',
 'අරහතො',
 'සම්මා',
 'සම්බුද්ධස්ස',
 'අප',
 'බුදුන්',
 'සාරාසකි',
 'කප්සුවහස්',
 'මතුයෙහි',
 'කුලුණුනු',
 'වණින්',
 'යුත්',
 'මහත',
 'වු',
 'සක්',
 'වැ',
 'දිවකුරු',
 'බුදුන්',
 'හමුවැ',
 'අතට',
 'පත්',
 'නිවන්',
 'සසර',
 'සේ',
 'පියා',
 'සතුන්',
 'සඳහා',
 'විඳුනා',
 'සසර',
 'දුක්',
 'නිවන්',
 'සේ',
 'ගෙන',
 'භව',
 'දුර්ගයට',
 'වැද',
 'පැරුම්',
 'පුරා',
 'තු',
 'පුර',
 'පැමිණ',
 'දසදහස්',
 'ලෝදාහි',
 'දෙව්බඹුන්ගේ',
 'අයජ',
 'මෙන්',
 'කිඹුල්වත්හි',
 'ඉපද',
 'වැඩිවිය',
 'පැමිණ',
 'මහභිනික්මන්',
 'කොට',
 'පැවිජි',
 'වැ',
 'මහා',
 'වීර්යය',
 'කොට',
 'බොධිපර්යංකාරූඪ',
 'වැ',
 'මාරවිජය',
 'කොට',
 'සර්වඥපදප්රාප්ත',
 'වැ',
 'හුනස්නෙන්',
 'නැඟී',
 'පූර්වොත්තර',
 'දිශා',
 'භාගයෙහි',
 'වැඩ',
 'සිට',
 'සප්තාහයක්',
 'අනිමිසලෝචන',
 'යුගලයෙන්',
 'පුදන',
 'ලද',
 'ජය',
 'මහා',
 'බොධිය',
 'තත',
 'ප්රභව',
 'වූ',
 'ඵල',
 'රුහ',
 'මහා',
 'බොධිය',
 'තදවයව',
 'වූ',
 'ශාඛා',
 'හා',
 'බොධිය',
 'යන',
 'තුන්',
 'මහා',
 'බොධීන්ගේ',
 'වංශයෙහි',
 'සහස්රරශ්මින්',
 '

In [38]:
# def load_stopwords(file_path):
#     try:
#         with open(file_path, 'r', encoding='utf-8') as f:
#             stopwords = f.read().splitlines()
#             # Remove empty lines and strip whitespace
#             stopwords = [word.strip() for word in stopwords if word.strip()]
#             return set(stopwords)  # set is faster for lookups
#     except Exception as e:
#         print(f"Error loading stopwords from {file_path}: {e}")
#         return set()

In [39]:
# sinhala_stopwords = load_stopwords("stop words.txt")

In [40]:
# sinhala_stopwords

In [41]:
# list_of_file_locations['filtered_tokens'] = list_of_file_locations['tokens'].apply(
#     lambda tokens: [word for word in tokens if word not in sinhala_stopwords]
# )

In [42]:
# all_tokens = sum(list_of_file_locations['filtered_tokens'], [])

In [43]:
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt

In [44]:
# counter = Counter(all_tokens)
# top_words = counter.most_common(100)

In [45]:
# top_words

In [46]:
# words, freqs = zip(*top_words)
# plt.figure(figsize=(12, 6))
# plt.bar(words, freqs, color='mediumseagreen')
# plt.xticks(rotation=45, ha='right', fontsize=13)
# plt.ylabel("Frequency")
# plt.title("Top 20 Sinhala Words")
# plt.tight_layout()
# plt.show()

In [47]:
# wordcloud = WordCloud(
#     font_path="NotoSerifSinhala-VariableFont_wdth.ttf",  # Make sure Sinhala font is available
#     background_color='white',
#     width=800,
#     height=400
# ).generate_from_frequencies(counter)

# plt.figure(figsize=(12, 6))
# plt.imshow(wordcloud, interpolation='bilinear')
# plt.axis('off')
# plt.title("Sinhala Word Cloud", fontsize=16)
# plt.show()

In [48]:
century_tokens = list_of_file_locations.groupby('end_century')['tokens'].sum()  # flatten lists
century_tokens = century_tokens.to_dict()

In [49]:
century_tokens

{5: ['සාරාර්ථ',
  'සංග්රඥාව',
  'තමස්ත',
  'භගවතෙහිත',
  'සම්යක්',
  'යමු',
  'බුද්ධාය',
  'නත්වා',
  'මුනින්දචරණං',
  'තිභවෙක',
  'සෙට්ඨං',
  'සන්තෙහි',
  'වුත්ත',
  'විවිධං',
  'වරතන්තසත්ථෙ',
  'අත්ථං',
  'හිසක්ක',
  'කුසලෙහි',
  'සමුචරිත්වා',
  'වක්ඛාමි',
  'සංගහ',
  'මං',
  'ජනසංග',
  'හත්ථං',
  'තිභවෙක',
  'සෙට්ඨං',
  'කාම',
  'ලොක',
  'රූප',
  'ලොක',
  'අරූපලෝක',
  'යන්',
  'ලොකත්රයට',
  'ශ්රෙෂ්ඨ',
  'වූ',
  'මුනින්ද',
  'චරණං',
  'මුනීන්ද්ර',
  'නම්',
  'වූ',
  'සම්යක්',
  'සම්බුද්ධයන්',
  'වහන්සේගේ',
  'ශ්රී',
  'පාද',
  'පද්මය',
  'නත්වා',
  'භය',
  'ලොහ',
  'කුලාචාරයෙන්',
  'විනා',
  'සකසා',
  'වැඳ',
  'සන්තෙහි',
  'සකල',
  'ශාස්ත්රයෙහි',
  'පාරප්රාප්ත',
  'පණ්ඩිත',
  'වූ',
  'භී',
  'සක්ක',
  'කුසලෙහි',
  'භෛෂජ්යාර්ථයෙහි',
  'දක්ෂ',
  'වූ',
  'වෙදුන්',
  'විසින්',
  'වර්ණ',
  'තන්තස්සත්ථෙ',
  'උතුම්',
  'වූ',
  'රස',
  'සංහිතාදි',
  'තන්ත්ර',
  'ශාස්ත්රයෙහි',
  'වුත',
  'කියන',
  'ලද',
  'විවිධං',
  'නානාප්රකාර',
  'වූ',
  'අත්ථං',
  'සාරාර්ථය',
  'සමුචරිත්වා',
  'උදුරාගෙණ',

In [50]:
len(century_tokens[5])

1062

In [51]:
len(list(set(century_tokens[5])))

775

In [52]:
len(century_tokens[13])

10932

In [53]:
len(list(set(century_tokens[13])))

5698

In [54]:
len(century_tokens[14])

5937

In [55]:
len(list(set(century_tokens[14])))

3212

In [56]:
len(century_tokens[15])

6607

In [57]:
len(list(set(century_tokens[15])))

3698

In [58]:
len(century_tokens[18])

3909

In [59]:
len(list(set(century_tokens[18])))

2152

In [60]:
len(century_tokens[19])

11214

In [61]:
len(list(set(century_tokens[19])))

6119

In [62]:
len(century_tokens[20])

18366

In [63]:
len(list(set(century_tokens[20])))

8171

In [64]:
# Get total frequency of each word per century
freq_by_century = {
    century: Counter(tokens)
    for century, tokens in century_tokens.items()
}

# Create a DataFrame: rows = words, columns = centuries
freq_df = pd.DataFrame(freq_by_century).fillna(0).astype(int)

In [65]:
freq_df

,5,13,14,15,18,19,20
සාරාර්ථ,4,0,0,0,0,0,1
සංග්රඥාව,1,0,0,0,0,0,0
තමස්ත,1,0,0,0,0,0,0
භගවතෙහිත,1,0,0,0,0,0,0
සම්යක්,2,1,7,0,2,5,2
...,...,...,...,...,...,...,...
අසලැ,0,0,0,0,0,0,1
නොතිබුණායින්,0,0,0,0,0,0,1
උදයැ,0,0,0,0,0,0,1
වැඩුයේ,0,0,0,0,0,0,1


In [66]:
freq_df.loc['නිදහසේ']

5     0
13    0
14    0
15    0
18    0
19    0
20    8
Name: නිදහසේ, dtype: int64

In [67]:
freq_df[20].describe()

count    22837.000000
mean         0.804221
std          3.799976
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max        236.000000
Name: 20, dtype: float64

In [68]:
zscore_df = freq_df.copy()
zscore_df = (freq_df - freq_df.mean()) / freq_df.std(ddof=0)

In [69]:
zscore_df.loc['නිදහසේ']

5    -0.132369
13   -0.184565
14   -0.141752
15   -0.142622
18   -0.129702
19   -0.239043
20    1.893679
Name: නිදහසේ, dtype: float64

In [70]:
zscore_df.head()

,5,13,14,15,18,19,20
සාරාර්ථ,11.253384,-0.184565,-0.141752,-0.142622,-0.129702,-0.239043,0.051522
සංග්රඥාව,2.714069,-0.184565,-0.141752,-0.142622,-0.129702,-0.239043,-0.211643
තමස්ත,2.714069,-0.184565,-0.141752,-0.142622,-0.129702,-0.239043,-0.211643
භගවතෙහිත,2.714069,-0.184565,-0.141752,-0.142622,-0.129702,-0.239043,-0.211643
සම්යක්,5.560507,0.200992,3.675042,-0.142622,1.385779,2.194979,0.314688


In [71]:
zscore_df.to_csv("Z_Scores.csv")

In [72]:
def compute_coverage(zscores, lower=-1.5, upper=1.5):
    total = zscores.size
    within = ((zscores > lower) & (zscores < upper)).sum().sum()
    return within / total

In [73]:
coverage = compute_coverage(zscore_df, lower=float('-inf'), upper=6.1027)

In [74]:
print(f"Coverage for z ∈ [-1.5, 1.5]: {coverage*100:.2f}%")

Coverage for z ∈ [-1.5, 1.5]: 99.80%


In [75]:
# typical_words = {
#     century: zscore_df[
#         zscore_df[century].between(-1.5, 1.5)
#     ].index.tolist()
#     for century in zscore_df.columns
# }

In [76]:
# typical_words

In [77]:
# len(typical_words[5])

In [78]:
distinctive_words = {
    century: zscore_df[
        ~zscore_df[century].between(float('-inf'), 6.1027)
    ].sort_values(by=century, ascending=False).index.tolist()
    for century in zscore_df.columns
}

In [79]:
distinctive_words

{5: ['හෝ',
  'වූ',
  'ලක්ෂණය',
  'යන',
  'මේ',
  'ද',
  'ව්යාධි',
  'නම්',
  'කීහ',
  'අංග',
  'දෙහ',
  'දොෂ',
  'ලක්ෂණ',
  'විසින්',
  'ගත',
  'දූත',
  'ව්යංග',
  'සම්පත්ය',
  'සාරාර්ථ',
  'දවසක්',
  'හට',
  'වෛද්ය',
  'රස',
  'දක්ෂ',
  'පාද',
  'භය',
  'දකුණු',
  'වීණා',
  'මොවුන්',
  'භෙරි',
  'පොතක',
  'සමහර',
  'ප්රශස්තෘ',
  'සකල',
  'සහිත',
  'තෙම',
  'එසේම',
  'තථා',
  'ලද',
  'වෙද',
  'ලක්ෂණම්',
  'ශාස්ත්රය',
  'කාල',
  'ප්රකෘති',
  'ආරොග්ය',
  'සත්ව',
  'උතුම්'],
 13: ['වූ',
  'නම්',
  'යන',
  'ච',
  'යැ',
  'ය',
  'වැ',
  'ඇති',
  'ති',
  'ද',
  'නැමැති',
  'සේ',
  'විසින්',
  'හෙයින්',
  'ඒ',
  'මේ',
  'මහා',
  'ලද',
  'වෙයි',
  'යි',
  'හා',
  'කොට',
  'වේ',
  'කල්හි',
  'නිවන්',
  'බුදුන්',
  'මහ',
  'නො',
  'හෝ',
  'බැවින්',
  'වන',
  'යනු',
  'වා',
  'මනා',
  'තන්හි',
  'න',
  'වහන්සේ',
  'බුදු',
  'සෙයින්',
  'තෙන',
  'වැනි',
  'යී'],
 14: ['වූ',
  'ඒ',
  'නම්',
  'කොට',
  'විසින්',
  'වහන්සේ',
  'ලද',
  'මේ',
  'ඇති',
  'හා',
  'කල්හි',
  'දෙසන',
  'දැක',
  'මහ',
  'මහ

In [80]:
distinctive_words[20]

['වූ',
 'ද',
 'ඒ',
 'මේ',
 'යන',
 'නම්',
 'යනු',
 'යි',
 'වේ',
 'නො',
 'කොට',
 'බව',
 'ලද',
 'ය',
 'විසින්',
 'එ',
 'ට',
 'ම',
 'යයි',
 'යුතු',
 'යැ',
 'මැ',
 'අ',
 'හා',
 'ආදි',
 'වන',
 'ඇති',
 'න',
 'මෙහි',
 'කරණ',
 'හෙයින්',
 'සේ',
 'වී',
 'ගේ',
 'මෙ',
 'මෙන්',
 'එක',
 'ආ',
 'ක්',
 'හෝ',
 'වෙයි',
 'ගුණ',
 'කියන',
 'පද',
 'ගත',
 'නා',
 'කර',
 'ර',
 'මහා',
 'කී',
 'කළ',
 'ච',
 'රජ',
 'එහි',
 'ක',
 'බැවින්',
 'වැ',
 'ව්යාකරණ',
 'යී',
 'කියා',
 'වා']

In [81]:
len(distinctive_words[5])

47

In [82]:
len(distinctive_words[13])

42

In [83]:
len(distinctive_words[14])

39

In [84]:
len(distinctive_words[15])

28

In [85]:
len(distinctive_words[18])

44

In [86]:
len(distinctive_words[19])

65

In [87]:
len(distinctive_words[20])

61

In [88]:
zscore_df.head()

,5,13,14,15,18,19,20
සාරාර්ථ,11.253384,-0.184565,-0.141752,-0.142622,-0.129702,-0.239043,0.051522
සංග්රඥාව,2.714069,-0.184565,-0.141752,-0.142622,-0.129702,-0.239043,-0.211643
තමස්ත,2.714069,-0.184565,-0.141752,-0.142622,-0.129702,-0.239043,-0.211643
භගවතෙහිත,2.714069,-0.184565,-0.141752,-0.142622,-0.129702,-0.239043,-0.211643
සම්යක්,5.560507,0.200992,3.675042,-0.142622,1.385779,2.194979,0.314688


In [89]:
zscore_dict_cen = zscore_df[19].to_dict()

In [90]:
word_scores = sorted(zscore_dict_cen.items(), key = lambda item: item[1], reverse=True)

In [91]:
with open("stopwords_19th.txt", "w", encoding="utf-8") as f:
    for word, score in word_scores:
        f.write(f"{word}\t{score:.4f}\n")

In [92]:
distinctive_words[5]

['හෝ',
 'වූ',
 'ලක්ෂණය',
 'යන',
 'මේ',
 'ද',
 'ව්යාධි',
 'නම්',
 'කීහ',
 'අංග',
 'දෙහ',
 'දොෂ',
 'ලක්ෂණ',
 'විසින්',
 'ගත',
 'දූත',
 'ව්යංග',
 'සම්පත්ය',
 'සාරාර්ථ',
 'දවසක්',
 'හට',
 'වෛද්ය',
 'රස',
 'දක්ෂ',
 'පාද',
 'භය',
 'දකුණු',
 'වීණා',
 'මොවුන්',
 'භෙරි',
 'පොතක',
 'සමහර',
 'ප්රශස්තෘ',
 'සකල',
 'සහිත',
 'තෙම',
 'එසේම',
 'තථා',
 'ලද',
 'වෙද',
 'ලක්ෂණම්',
 'ශාස්ත්රය',
 'කාල',
 'ප්රකෘති',
 'ආරොග්ය',
 'සත්ව',
 'උතුම්']

In [93]:
first_10_words_per_century = {
    century: words[:10] for century, words in distinctive_words.items()
}

In [94]:
first_10_words_per_century

{5: ['හෝ', 'වූ', 'ලක්ෂණය', 'යන', 'මේ', 'ද', 'ව්යාධි', 'නම්', 'කීහ', 'අංග'],
 13: ['වූ', 'නම්', 'යන', 'ච', 'යැ', 'ය', 'වැ', 'ඇති', 'ති', 'ද'],
 14: ['වූ', 'ඒ', 'නම්', 'කොට', 'විසින්', 'වහන්සේ', 'ලද', 'මේ', 'ඇති', 'හා'],
 15: ['වූ', 'කොට', 'ර', 'ද', 'ස', 'මේ', 'හා', 'නම්', 'ලද', 'ත'],
 18: ['වූ',
  'යන',
  'වේ',
  'කියාවේ',
  'තන්හි',
  'නම්',
  'සූත්රයෙන්',
  'මේ',
  'හට',
  'කියා'],
 19: ['වූ', 'ඒ', 'නම්', 'න', 'ද', 'මේ', 'කළ', 'කොට', 'විසින්', 'කරණ'],
 20: ['වූ', 'ද', 'ඒ', 'මේ', 'යන', 'නම්', 'යනු', 'යි', 'වේ', 'නො']}

In [95]:
union_words = {word for words_list in distinctive_words.values() for word in words_list}

In [96]:
sorted_union = sorted(
    union_words,
    key=lambda word: zscore_df.loc[word].mean(),
    reverse=True
)

In [97]:
sorted_union

['වූ',
 'නම්',
 'යන',
 'ඒ',
 'මේ',
 'ද',
 'කොට',
 'විසින්',
 'ඇති',
 'ලද',
 'වේ',
 'හෝ',
 'හා',
 'න',
 'යනු',
 'ච',
 'ය',
 'වහන්සේ',
 'කළ',
 'හට',
 'හෙයින්',
 'ලක්ෂණය',
 'මහා',
 'බව',
 'යයි',
 'යි',
 'කියා',
 'කරණ',
 'කල්හි',
 'සේ',
 'ස',
 'ති',
 'යැ',
 'මහ',
 'ර',
 'වන',
 'තන්හි',
 'එක',
 'තෙම',
 'නො',
 'මෙන්',
 'එහි',
 'පිණිස',
 'බුද්ධ',
 'ගුණ',
 'වැ',
 'මෙහි',
 'කර',
 'රජ',
 'මා',
 'ට',
 'ම',
 'පූර්ව',
 'වෙයි',
 'උතුම්',
 'කීහ',
 'යුතු',
 'වා',
 'දැක',
 'ගත',
 'බුදුන්',
 'ක',
 'සඳ',
 'ධර්ම',
 'දී',
 'නැමැති',
 'කියන',
 'ලක්ෂණ',
 'කී',
 'පද',
 'සකල',
 'සූත්රයෙන්',
 'විය',
 'කියාවේ',
 'වැනි',
 'සිටි',
 'ශ්රී',
 'වී',
 'මෙ',
 'නැති',
 'සහ',
 'තං',
 'අ',
 'එ',
 'ගේ',
 'හි',
 'රස',
 'වැඩ',
 'එම',
 'සහිත',
 'ලෙස',
 'මම',
 'තෙමේ',
 'හැර',
 'කියාද',
 'ආදි',
 'ත',
 'බැවින්',
 'ණ',
 'ඉතා',
 'හෙවත්',
 'ල',
 'පැමිණ',
 'අංග',
 'ඉති',
 'ව්යාධි',
 'සර්වඥයන්',
 'කරා',
 'නමැති',
 'සේක',
 'නුවර',
 'පමණ',
 'ලකුණු',
 'මෙම',
 'ගෙණ',
 'මැ',
 'එසේම',
 'යී',
 'කරන',
 'පන',
 'තැන',
 'තථා',
 'පාද',
 'ඔහු',
 

In [98]:
len(sorted_union)

194

In [99]:
data_for_df = {}

for century, words_list in distinctive_words.items():
    availability_list = []
    for word in sorted_union:
        if word in words_list:
            availability_list.append(1)  # 1 for present
        else:
            availability_list.append(0)  # 0 for not present
    
    # Add the list of 1s and 0s to the dictionary
    data_for_df[century] = availability_list

In [100]:
availability_df = pd.DataFrame(data_for_df, index=sorted_union)

In [101]:
availability_df

,5,13,14,15,18,19,20
වූ,1,1,1,1,1,1,1
නම්,1,1,1,1,1,1,1
යන,1,1,1,1,1,1,1
ඒ,0,1,1,0,1,1,1
මේ,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...
වෘත්ත,0,0,0,0,1,0,0
ලද්දේද,0,0,1,0,0,0,0
ව්යාකරණ,0,0,0,0,0,0,1
සුත්රයෙන්,0,0,0,0,1,0,0


In [102]:
availability_df.to_csv('Stopwords_Distribution.csv', index=True)